<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DataWrangling/blob/main/W04_Word_Extraktion_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Informationsextraktion aus Word-Dateien
**Modul: Data Engineering and Wrangling | FHNW Hochschule für Wirtschaft | FS 2026**

---

In diesem Notebook lernst du, wie man Informationen aus `.docx`-Dateien mit Python extrahiert.  
Wir verwenden die Bibliothek `python-docx` und üben anhand konkreter Aufgaben.

### Inhalt
1. [Installation & Setup](#1-installation--setup)
2. [Struktur einer .docx-Datei](#2-struktur-einer-docx-datei)
3. [Absätze & Texte lesen](#3-absätze--texte-lesen)
4. [Formatierungen & Styles](#4-formatierungen--styles)
5. [Tabellen extrahieren](#5-tabellen-extrahieren)
6. [Metadaten auslesen](#6-metadaten-auslesen)
7. [Bilder extrahieren](#7-bilder-extrahieren)
8. [Übungsaufgaben](#8-übungsaufgaben)


---
## 1. Installation & Setup


In [ ]:
# Bibliothek installieren (nur einmal nötig)
!pip install python-docx

In [ ]:
from docx import Document
from docx.shared import Inches, Pt
import pandas as pd
import os
import json
from pathlib import Path

print('✅ Alle Bibliotheken erfolgreich importiert!')
print(f'python-docx Version:', __import__('docx').__version__)

---
## 2. Struktur einer .docx-Datei

Eine `.docx`-Datei ist ein **ZIP-Archiv** mit XML-Dateien. Die wichtigsten Dateien darin:
- `word/document.xml` → Der gesamte Textinhalt
- `word/styles.xml` → Alle Formatvorlagen  
- `word/media/` → Eingebettete Bilder
- `docProps/core.xml` → Metadaten (Autor, Datum, ...)

**Tipp:** Benenne eine `.docx`-Datei in `.zip` um und öffne sie mit einem Archivprogramm!


In [ ]:
# Zuerst eine Test-Word-Datei erstellen (damit du keine eigene brauchst)
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime

def create_sample_document():
    """Erstellt ein Beispiel-Word-Dokument für die Übungen."""
    doc = Document()

    # Metadaten setzen
    doc.core_properties.title = 'Quartalsbericht Q1 2026'
    doc.core_properties.author = 'Max Muster'
    doc.core_properties.subject = 'Data Engineering Beispiel'
    doc.core_properties.keywords = 'Bericht, Daten, FHNW'
    doc.core_properties.description = 'Beispieldokument für python-docx Übungen'

    # Überschrift
    doc.add_heading('Quartalsbericht Q1 2026', level=0)
    doc.add_heading('1. Zusammenfassung', level=1)
    doc.add_paragraph(
        'Dieser Bericht gibt einen Überblick über die Datenverarbeitungsaktivitäten '
        'im ersten Quartal 2026. Die Gesamtverarbeitung umfasste 42.350 Datensätze.'
    )

    doc.add_heading('2. Kennzahlen', level=1)
    p = doc.add_paragraph('Wichtige Kennzahlen: ')
    run = p.add_run('42.350 Datensätze')
    run.bold = True
    p.add_run(' wurden erfolgreich verarbeitet.')

    # Aufzählung
    doc.add_paragraph('Verarbeitungsrate: 98.7%', style='List Bullet')
    doc.add_paragraph('Fehlerquote: 1.3%', style='List Bullet')
    doc.add_paragraph('Durchlaufzeit: 4.2 Sekunden/Datensatz', style='List Bullet')

    doc.add_heading('3. Umsatzdaten nach Abteilung', level=1)
    doc.add_paragraph('Die folgende Tabelle zeigt die Umsätze nach Abteilung:')

    # Tabelle hinzufügen
    table = doc.add_table(rows=1, cols=4)
    table.style = 'Table Grid'

    # Kopfzeile
    headers = ['Abteilung', 'Q1 2025', 'Q1 2026', 'Änderung']
    for i, h in enumerate(headers):
        table.rows[0].cells[i].text = h

    # Datenzeilen
    data = [
        ['Vertrieb', '245.000 CHF', '287.500 CHF', '+17.3%'],
        ['Marketing', '98.000 CHF', '112.000 CHF', '+14.3%'],
        ['IT', '145.000 CHF', '158.000 CHF', '+9.0%'],
        ['Gesamt', '488.000 CHF', '557.500 CHF', '+14.2%'],
    ]
    for row_data in data:
        row = table.add_row()
        for i, val in enumerate(row_data):
            row.cells[i].text = val

    doc.add_heading('4. Fazit', level=1)
    doc.add_paragraph(
        'Das erste Quartal 2026 verlief positiv. Die Datenqualität konnte '
        'durch neue Bereinigungsprozesse deutlich verbessert werden.'
    )

    doc.save('beispiel_bericht.docx')
    print('✅ Beispieldokument erstellt: beispiel_bericht.docx')
    return doc

create_sample_document()

In [ ]:
# Dokument laden und Grundstruktur erkunden
doc = Document('beispiel_bericht.docx')

print('=== Dokumentstruktur ===')
print(f'Anzahl Absätze:  {len(doc.paragraphs)}')
print(f'Anzahl Tabellen: {len(doc.tables)}')
print(f'Anzahl Sections: {len(doc.sections)}')

---
## 3. Absätze & Texte lesen


In [ ]:
# Alle Absätze mit ihrem Style-Namen ausgeben
doc = Document('beispiel_bericht.docx')

print('=== Alle Absätze mit Styles ===')
for i, para in enumerate(doc.paragraphs):
    if para.text.strip():  # Leere Absätze überspringen
        print(f'[{para.style.name:20s}] {para.text[:60]}')

In [ ]:
# Nur Überschriften extrahieren
headings = [
    {'level': para.style.name, 'text': para.text}
    for para in doc.paragraphs
    if para.style.name.startswith('Heading') and para.text.strip()
]

print('=== Inhaltsverzeichnis ===')
for h in headings:
    level = h['level'].replace('Heading ', '')
    indent = '  ' * (int(level) - 1) if level.isdigit() else ''
    print(f'{indent}► {h["text"]}')

In [ ]:
# Gesamten Text extrahieren (eine Zeile pro Absatz)
full_text = '\n'.join(
    para.text for para in doc.paragraphs if para.text.strip()
)
print(full_text[:500], '...')

---
## 4. Formatierungen & Styles

Jeder Absatz besteht aus **Runs** – Textfragmente mit einheitlicher Formatierung.


In [ ]:
# Formatierungen auf Run-Ebene analysieren
print('=== Formatierte Textstellen ===')
for para in doc.paragraphs:
    for run in para.runs:
        if run.text.strip():
            flags = []
            if run.bold:   flags.append('FETT')
            if run.italic: flags.append('KURSIV')
            if run.underline: flags.append('UNTERSTRICHEN')
            if flags:
                print(f'[{", ".join(flags)}] "{run.text}"')

In [ ]:
# Alle verfügbaren Styles im Dokument anzeigen
styles_used = {para.style.name for para in doc.paragraphs if para.text.strip()}
print('Verwendete Styles:')
for style in sorted(styles_used):
    count = sum(1 for p in doc.paragraphs if p.style.name == style and p.text.strip())
    print(f'  {style:25s}: {count} Absatz/Absätze')

---
## 5. Tabellen extrahieren


In [ ]:
# Erste Tabelle → pandas DataFrame
def table_to_dataframe(table):
    """Konvertiert eine python-docx Tabelle in einen pandas DataFrame."""
    # Kopfzeile (erste Zeile)
    headers = [cell.text.strip() for cell in table.rows[0].cells]

    # Datenzeilen
    data = []
    for row in table.rows[1:]:
        row_data = [cell.text.strip() for cell in row.cells]
        data.append(row_data)

    return pd.DataFrame(data, columns=headers)


# Alle Tabellen verarbeiten
print(f'Gefundene Tabellen: {len(doc.tables)}\n')

for i, table in enumerate(doc.tables):
    df = table_to_dataframe(table)
    print(f'--- Tabelle {i + 1} ({len(table.rows)} Zeilen, {len(table.columns)} Spalten) ---')
    print(df)
    print()

In [ ]:
# Tabelle als CSV exportieren
if doc.tables:
    df = table_to_dataframe(doc.tables[0])
    df.to_csv('umsatzdaten.csv', index=False)
    print('✅ Tabelle exportiert nach: umsatzdaten.csv')
    print(df)

---
## 6. Metadaten auslesen


In [ ]:
# Alle Metadaten auslesen
def extract_metadata(filepath):
    """Extrahiert alle verfügbaren Metadaten aus einer Word-Datei."""
    doc = Document(filepath)
    props = doc.core_properties

    metadata = {
        'titel':            props.title,
        'autor':            props.author,
        'thema':            props.subject,
        'stichwörter':      props.keywords,
        'erstellt':         str(props.created) if props.created else None,
        'geändert':         str(props.modified) if props.modified else None,
        'zuletzt_von':      props.last_modified_by,
        'sprache':          props.language,
        'revision':         props.revision,
        'anzahl_absätze':   len(doc.paragraphs),
        'anzahl_tabellen':  len(doc.tables),
        'wortanzahl_schätzung': sum(
            len(p.text.split()) for p in doc.paragraphs
        )
    }
    return metadata


meta = extract_metadata('beispiel_bericht.docx')
print('=== Metadaten ===')
for key, value in meta.items():
    print(f'  {key:25s}: {value}')

In [ ]:
# Metadaten als JSON speichern
with open('metadaten.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print('✅ Metadaten gespeichert: metadaten.json')

---
## 7. Bilder extrahieren


In [ ]:
def extract_images(filepath, output_dir='extracted_images'):
    """
    Extrahiert alle eingebetteten Bilder aus einer Word-Datei.
    Gibt eine Liste der gespeicherten Dateipfade zurück.
    """
    doc = Document(filepath)
    os.makedirs(output_dir, exist_ok=True)
    saved_files = []

    for i, rel in enumerate(doc.part.rels.values()):
        if 'image' in rel.reltype:
            img_data = rel.target_part.blob
            content_type = rel.target_part.content_type
            extension = content_type.split('/')[-1]
            # PNG und JPEG normalisieren
            if extension == 'jpeg':
                extension = 'jpg'

            filename = f'bild_{i:03d}.{extension}'
            filepath_out = os.path.join(output_dir, filename)

            with open(filepath_out, 'wb') as f:
                f.write(img_data)

            saved_files.append({
                'datei': filename,
                'format': content_type,
                'größe_kb': len(img_data) / 1024
            })

    return saved_files


bilder = extract_images('beispiel_bericht.docx')
if bilder:
    print(f'✅ {len(bilder)} Bild(er) extrahiert:')
    for b in bilder:
        print(f'  {b["datei"]:20s} | Format: {b["format"]:20s} | {b["größe_kb"]:.1f} KB')
else:
    print('ℹ️ Keine Bilder im Dokument gefunden (unser Beispieldokument hat keine).')

---
## 8. Übungsaufgaben

> 💡 **Hinweis:** Verwende für alle Aufgaben das Beispieldokument `beispiel_bericht.docx` oder ein eigenes `.docx`-Dokument.


### Aufgabe 1 – Einfach: Dokument erkunden 🟢

**Aufgabe:** Lade das Beispieldokument und beantworte folgende Fragen mit Code:
1. Wie viele Absätze hat das Dokument?
2. Wie viele davon sind **nicht leer**?
3. Welche Style-Namen kommen vor? (als Liste, keine Duplikate)
4. Wie viele Tabellen hat das Dokument?


In [ ]:
# 📝 Deine Lösung für Aufgabe 1
doc = Document('beispiel_bericht.docx')

# 1. Anzahl Absätze
# ...

# 2. Nicht-leere Absätze
# ...

# 3. Unique Styles
# ...

# 4. Anzahl Tabellen
# ...


In [ ]:
# ✅ Musterlösung Aufgabe 1 (nur zur Kontrolle!)
doc = Document('beispiel_bericht.docx')

# 1.
print(f'1. Alle Absätze: {len(doc.paragraphs)}')

# 2.
nicht_leer = [p for p in doc.paragraphs if p.text.strip()]
print(f'2. Nicht-leere Absätze: {len(nicht_leer)}')

# 3.
styles = sorted({p.style.name for p in doc.paragraphs if p.text.strip()})
print(f'3. Style-Namen: {styles}')

# 4.
print(f'4. Anzahl Tabellen: {len(doc.tables)}')

### Aufgabe 2 – Mittel: Strukturierter Export 🟡

**Aufgabe:** Schreibe eine Funktion `strukturierter_export(doc)`, die ein Dictionary zurückgibt mit der Struktur:
```python
{
  'Heading 1': ['Text des ersten Abschnitts...', ...],
  'Heading 2': [...],
  'Normal': [...],
  ...
}
```
Speichere das Ergebnis als `struktur.json`.


In [ ]:
# 📝 Deine Lösung für Aufgabe 2
def strukturierter_export(doc):
    """Gibt Texte gruppiert nach Style-Namen zurück."""
    ergebnis = {}
    # Dein Code hier...
    return ergebnis

doc = Document('beispiel_bericht.docx')
struktur = strukturierter_export(doc)
# Ausgabe & JSON-Export


In [ ]:
# ✅ Musterlösung Aufgabe 2
def strukturierter_export(doc):
    ergebnis = {}
    for para in doc.paragraphs:
        if para.text.strip():
            style = para.style.name
            ergebnis.setdefault(style, []).append(para.text)
    return ergebnis

doc = Document('beispiel_bericht.docx')
struktur = strukturierter_export(doc)

for style, texte in struktur.items():
    print(f'\n[{style}]')
    for t in texte:
        print(f'  → {t[:70]}')

with open('struktur.json', 'w', encoding='utf-8') as f:
    json.dump(struktur, f, ensure_ascii=False, indent=2)
print('\n✅ Gespeichert: struktur.json')

### Aufgabe 3 – Mittel: Alle Tabellen als CSV exportieren 🟡

**Aufgabe:** Schreibe eine Funktion `alle_tabellen_exportieren(filepath)` die:
1. Alle Tabellen im Dokument liest
2. Jede Tabelle in einen DataFrame konvertiert
3. Jede Tabelle als `tabelle_1.csv`, `tabelle_2.csv`, etc. speichert
4. Eine Zusammenfassung (Anzahl Tabellen, Zeilen, Spalten) ausgibt


In [ ]:
# 📝 Deine Lösung für Aufgabe 3
def alle_tabellen_exportieren(filepath):
    doc = Document(filepath)
    # Dein Code hier...
    pass

alle_tabellen_exportieren('beispiel_bericht.docx')

In [ ]:
# ✅ Musterlösung Aufgabe 3
def alle_tabellen_exportieren(filepath):
    doc = Document(filepath)
    print(f'Gefunden: {len(doc.tables)} Tabelle(n)\n')

    for i, table in enumerate(doc.tables, start=1):
        headers = [cell.text.strip() for cell in table.rows[0].cells]
        data = [[cell.text.strip() for cell in row.cells]
                for row in table.rows[1:]]
        df = pd.DataFrame(data, columns=headers)

        csv_name = f'tabelle_{i}.csv'
        df.to_csv(csv_name, index=False)
        print(f'  Tabelle {i}: {len(df)} Zeilen × {len(df.columns)} Spalten → {csv_name}')
        print(df, '\n')

alle_tabellen_exportieren('beispiel_bericht.docx')

### Aufgabe 4 – Schwer: Batch-Verarbeitung 🔴

**Aufgabe:** Erstelle zuerst 3 leicht unterschiedliche Testdokumente, dann schreibe eine Funktion `batch_analyse(ordner)` die:
- Alle `.docx`-Dateien in einem Ordner liest
- Für jede Datei: Titel, Autor, Wortanzahl, Anzahl Tabellen extrahiert
- Alles in einem übersichtlichen DataFrame zusammenfasst


In [ ]:
# Schritt 1: Test-Ordner mit mehreren Dokumenten erstellen
os.makedirs('test_dokumente', exist_ok=True)

for i in range(1, 4):
    d = Document()
    d.core_properties.title = f'Bericht {i}'
    d.core_properties.author = f'Autor {i}'
    d.add_heading(f'Testdokument {i}', level=0)
    d.add_paragraph(f'Inhalt des Berichts {i}. ' * (i * 5))
    if i > 1:
        t = d.add_table(rows=i, cols=2)
        t.style = 'Table Grid'
        for row in t.rows:
            row.cells[0].text = f'Schlüssel {i}'
            row.cells[1].text = f'Wert {i}'
    d.save(f'test_dokumente/bericht_{i}.docx')

print('✅ 3 Testdokumente in Ordner "test_dokumente" erstellt')

In [ ]:
# 📝 Deine Lösung für Aufgabe 4
def batch_analyse(ordner):
    """Analysiert alle .docx Dateien in einem Ordner."""
    ergebnisse = []
    # Dein Code hier...
    return pd.DataFrame(ergebnisse)

df_batch = batch_analyse('test_dokumente')
print(df_batch)

In [ ]:
# ✅ Musterlösung Aufgabe 4
def batch_analyse(ordner):
    ergebnisse = []
    for pfad in Path(ordner).glob('*.docx'):
        try:
            doc = Document(pfad)
            props = doc.core_properties
            wörter = sum(len(p.text.split()) for p in doc.paragraphs)
            ergebnisse.append({
                'datei':       pfad.name,
                'titel':       props.title or '(kein Titel)',
                'autor':       props.author or '(unbekannt)',
                'absätze':     len(doc.paragraphs),
                'tabellen':    len(doc.tables),
                'wortanzahl':  wörter,
                'erstellt':    str(props.created)[:10] if props.created else None,
            })
        except Exception as e:
            print(f'Fehler bei {pfad.name}: {e}')
    return pd.DataFrame(ergebnisse)

df_batch = batch_analyse('test_dokumente')
print('=== Batch-Analyse ===')
print(df_batch.to_string(index=False))

---
## 🎓 Zusammenfassung

| Aufgabe | Methode | Beschreibung |
|---|---|---|
| Laden | `Document(pfad)` | Word-Datei öffnen |
| Absätze | `doc.paragraphs` | Liste aller Absätze |
| Style | `para.style.name` | Formatvorlage |
| Formatierung | `run.bold`, `run.italic` | Run-Ebene |
| Tabellen | `doc.tables[i].rows[j].cells[k].text` | Tabelleninhalt |
| Metadaten | `doc.core_properties` | Autor, Datum, Titel |
| Bilder | `doc.part.rels` | Eingebettete Bilder |

**Weiterführend:** Schau dir die [python-docx Dokumentation](https://python-docx.readthedocs.io) an!
